# Imports

In [1]:
import requests
from bs4 import BeautifulSoup
import regex as re
import pandas as pd

# Proof of Concept

In [ ]:
r = requests.get("https://mtgtop8.com/format?f=ST&meta=46&cp=2")
print(r.status_code)

In [ ]:
soup = BeautifulSoup(r.content)
event_table = soup.find_all("table")[2]   # third table on page
event_table

In [ ]:
# get the event links
event_links_raw = re.findall("<a href=.+?>", str(event_table))
event_links_clean = [link.split("\"")[1] for link in event_links_raw]
event_links_clean

In [ ]:
# get the event name and date, not sure how i want to structure this atm 
events_raw = event_table.get_text()
events = re.split("\\n{3,4}", events_raw)[1:]
for event in events:
    event_name, event_date = re.split("\\n{2}", event)[:2]
    print(event_name, '\t', event_date)
#
# [re.split("\\n{2}", event)[:2] for event in events]

# Getting All Tournaments

Tournaments for the last two months are listed 20 at a time across 6 pages, so we need to get tournaments table on each of these pages.

In [2]:
# get all tournaments html
tournaments_tables = []
for page in range(1, 7):
    r = requests.get(f"https://mtgtop8.com/format?f=ST&meta=46&cp={page}")
    if r.status_code != 200:
        print(f"Could not scrape events from page {page}.")
        continue
    #

    soup = BeautifulSoup(r.content)
    tournaments_tables.append(soup.find_all("table")[2])   # third table on page 
#
display(tournaments_tables)

[<table align="center" border="0" class="Stable" width="98%">
 <tr class="hover_tr">
 <td align="center" width="5%"><img height="14" src="/graph/online/mtgo.png" title="MTG Online"/></td>
 <td class="S14" width="70%"><a href="event?e=80372&amp;f=ST">MTGO Challenge 32</a> <span class="new">NEW</span></td>
 <td align="center" width="13%"><img src="/graph/star.png"/><img src="/graph/star.png"/></td>
 <td align="right" class="S12" width="12%">13/02/26</td>
 </tr>
 <tr class="hover_tr">
 <td align="center" width="5%"><img height="14" src="/graph/online/mtgo.png" title="MTG Online"/></td>
 <td class="S14" width="70%"><a href="event?e=80328&amp;f=ST">MTGO Challenge 32</a> <span class="new">NEW</span></td>
 <td align="center" width="13%"><img src="/graph/star.png"/><img src="/graph/star.png"/></td>
 <td align="right" class="S12" width="12%">12/02/26</td>
 </tr>
 <tr class="hover_tr">
 <td align="center" width="5%"><img height="14" src="/graph/online/mtgo.png" title="MTG Online"/></td>
 <td cla

Now that we have the html for all tournaments tables, we need to extract the following information:
- tournament name
- tournament date
- link to tournament information

In [5]:
names = []
dates = []
links = []
for tournaments_table in tournaments_tables:
    # get event names and dates
    table_text = tournaments_table.get_text()
    name_dates = re.split("\\n{3,4}|\\n{2}", table_text)[1:]
    names.extend(name_dates[:-1:2])
    dates.extend(name_dates[1::2])

    # get event links
    table_hrefs = re.findall("<a href=.+?>", str(tournaments_table))
    links.extend([re.split("\"", str(table_href))[1] for table_href in table_hrefs])
#
all_tournaments_df = pd.DataFrame({'Name': names, 'Date': dates, 'Link': links})
all_tournaments_df["Date"] = pd.to_datetime(all_tournaments_df["Date"], format="%d/%m/%y")
display(all_tournaments_df, all_tournaments_df.dtypes)

,Name,Date,Link
0,MTGO Challenge 32 NEW,2026-02-13,event?e=80372&amp;f=ST
1,MTGO Challenge 32 NEW,2026-02-12,event?e=80328&amp;f=ST
2,MTGO Challenge 32 NEW,2026-02-10,event?e=80293&amp;f=ST
3,MTGO Challenge 64,2026-02-09,event?e=80250&amp;f=ST
4,MTGO Challenge 32,2026-02-08,event?e=80147&amp;f=ST
...,...,...,...
112,"RCQ @ SpellCrafter's Den (Upper Sandusky, OH)",2025-12-20,event?e=78209&amp;f=ST
113,"Champions Cup Special Qualifier @ TC (Osaka, J...",2025-12-20,event?e=78207&amp;f=ST
114,MTGO Challenge 32,2025-12-20,event?e=78173&amp;f=ST
115,MTGO Challenge 32,2025-12-20,event?e=78134&amp;f=ST


Name               str
Date    datetime64[us]
Link               str
dtype: object

The dataframe above has *all* large tournaments in the last two months. Some of these tournaments include Magic: The Gathering games played online in the app which has an alternative system for pricing cards. Due to this we want to only look at tournaments played outside of the app. 

Thankfully, these tournaments are denoted with "MTGO" so we can filter the `Name` property of the `tournaments_df` to only include tournaments without "MTGO" in the name. 

Also, we are only concerned with tournaments in January 2026, so we'll filter tournaments from before or after that time out as well. 

Lastly, the `Link` contains only the suffix of the URL. We need to append the base URL (mtgtop8.com) to this. 

In [11]:
# all tournaments that are not online (i.e., do not include "MTGO")
not_online = ~all_tournaments_df["Name"].str.contains("MTGO")

# all tournaments in january
in_january = (all_tournaments_df["Date"] >= '2026-01-01') & (all_tournaments_df["Date"] < '2026-02-01')

tournaments_df = all_tournaments_df[not_online & in_january].reset_index(drop=True)

# adding base_url to links
base_url = "https://mtgtop8.com"
tournaments_df["Link"] = base_url + '/' + tournaments_df["Link"]

tournaments_df

,Name,Date,Link
0,2nd Chance PTQ @ Pro Tour Lorwyn Eclipsed (Ric...,2026-01-31,https://mtgtop8.com/event?e=79854&amp;f=ST
1,Champions Cup Special Qualifier @ Kawasaki (Ja...,2026-01-30,https://mtgtop8.com/event?e=79785&amp;f=ST
2,Pro Tour Lorwyn Eclipsed @ Richmond,2026-01-30,https://mtgtop8.com/event?e=79746&amp;f=ST
3,Saturday ReCQ #2 @ SCG CON Portland,2026-01-25,https://mtgtop8.com/event?e=79664&amp;f=ST
4,Super Sunday RCQ #2 @ SCG CON Portland,2026-01-25,https://mtgtop8.com/event?e=79600&amp;f=ST
5,Super Sunday RCQ #1 @ SCG CON Portland,2026-01-25,https://mtgtop8.com/event?e=79599&amp;f=ST
6,Izzet Explosive Experiment Event,2026-01-25,https://mtgtop8.com/event?e=79598&amp;f=ST
7,Regional Championship @ SCG CON Portland,2026-01-25,https://mtgtop8.com/event?e=79462&amp;f=ST
8,Saturday RCQ @ SCG CON Portland,2026-01-24,https://mtgtop8.com/event?e=79569&amp;f=ST
9,"RCQ @ BB-Spiele (Rosenheim, Germany)",2026-01-24,https://mtgtop8.com/event?e=79428&amp;f=ST


All in all, we're left with 44 tournaments to grab decks for!

# Getting Decklists for Each Tournament

## Test

In [12]:
r = requests.get("https://mtgtop8.com/event?e=79854&amp;f=ST")
print(r.status_code)

200


In [27]:
soup = BeautifulSoup(r.content)
# print(soup.prettify())
# soup.find_all("div", class_="chosen_tr")
test = soup.find_all("a", class_="player")
for t in test:
    display(t.get_text())

'Collins Mullen'

'Edgar Magalhaes'

'Guiyote'

'Thomas Mechin'

'Ondrej Strasky'

...

'Sebastian Sachse'

'Ian Starkebaum'

'James Newman'

'David Gonzalez Romero "playmobil"'

'Steven Browne'

In [42]:
# you can either do this and try to filter down with regex via tags or do the commented method
test = re.findall("<a href=\"\?e=79854.+?>.+?</a>", str(soup)) # or <a href=\"\?e=79854.+?>[^<|>]+</a>
test

<>:2: SyntaxWarning: invalid escape sequence '\?'
<>:2: SyntaxWarning: invalid escape sequence '\?'
/var/folders/mc/k7dtvknn215g0s76b8mlhf_r0000gp/T/ipykernel_35854/4140714637.py:2: SyntaxWarning: invalid escape sequence '\?'
  test = re.findall("<a href=\"\?e=79854.+?>.+?</a>", str(soup)) # or <a href=\"\?e=79854.+?>[^<|>]+</a>


['<a href="?e=79854&amp;d=806938&amp;f=ST"><img src="/metas_thumbs/808.jpg"/></a>',
 '<a href="?e=79854&amp;d=806938&amp;f=ST">Temur Full Bore Harmonizer </a>',
 '<a href="?e=79854&amp;d=806939&amp;f=ST"><img src="/metas_thumbs/13.jpg"/></a>',
 '<a href="?e=79854&amp;d=806939&amp;f=ST">Dimir Control</a>',
 '<a href="?e=79854&amp;d=806941&amp;f=ST"><img src="/metas_thumbs/2843.jpg"/></a>',
 '<a href="?e=79854&amp;d=806941&amp;f=ST">Bant Airbending</a>',
 '<a href="?e=79854&amp;d=806940&amp;f=ST"><img src="/metas_thumbs/15.jpg"/></a>',
 '<a href="?e=79854&amp;d=806940&amp;f=ST">Izzet Spellementals</a>',
 '<a href="?e=79854&amp;d=806942&amp;f=ST"><img src="/metas_thumbs/15.jpg"/></a>',
 '<a href="?e=79854&amp;d=806942&amp;f=ST">Izzet Lesson</a>',
 '<a href="?e=79854&amp;d=806945&amp;f=ST"><img src="/metas_thumbs/15.jpg"/></a>',
 '<a href="?e=79854&amp;d=806945&amp;f=ST">Izzet Lesson</a>',
 '<a href="?e=79854&amp;d=806943&amp;f=ST"><img src="/metas_thumbs/832.jpg"/></a>',
 '<a href="?e=798

# Some Statistics (3-5)

# Some Viz 

Only base card price. Foil etc prices are for looks and not for usability. base card price is only about usability. 